# Customer Shopping Behavior: Data Cleaning and Feature Engineering

Turns the raw CSV into the cleaned table that is loaded into SQL Server (`dbo.customer_shopping`).

**Steps**
1. Load raw data
2. Rename columns to SQL-friendly names
3. Fill missing review ratings
4. Drop the duplicate `promo_code_used` column
5. Convert purchase frequency to number of days
6. Create `age_group`
7. Check the result
8. Load into SQL Server

In [ ]:
import pandas as pd

RAW_FILE = 'customer_shopping_behavior.csv'   # change path if needed
df = pd.read_csv(RAW_FILE)
print(df.shape)
df.head()

## 1. Quick look at the raw data

In [ ]:
df.info()
print()
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('Duplicate customer IDs:', df['Customer ID'].duplicated().sum())

## 2. Rename columns
Lower case, underscores, no brackets or spaces, so SQL queries are easy to write.

In [ ]:
df.columns = (
    df.columns.str.lower()
              .str.replace(' ', '_')
)
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})
df.columns.tolist()

## 3. Fill missing review ratings
149 ratings are missing. Fill each one with the **median rating of its category**, so a missing value in Footwear gets the Footwear median and not one number for everything.

In [ ]:
print('Missing before:', df['review_rating'].isnull().sum())

df['review_rating'] = df.groupby('category')['review_rating'].transform(
    lambda s: s.fillna(s.median())
)

print('Missing after :', df['review_rating'].isnull().sum())

## 4. Drop `promo_code_used`
It has the same value as `discount_applied` in every row, so it adds nothing.

In [ ]:
print('Identical to discount_applied:', (df['promo_code_used'] == df['discount_applied']).all())
df = df.drop(columns='promo_code_used')

## 5. Convert purchase frequency to days
Text like "Quarterly" cannot be sorted or averaged. Days between purchases can.

`Weekly` = 7, `Bi-Weekly` and `Fortnightly` = 14, `Monthly` = 30, `Quarterly` and `Every 3 Months` = 90, `Annually` = 365.

In [ ]:
freq_days = {
    'Weekly': 7,
    'Bi-Weekly': 14,
    'Fortnightly': 14,
    'Monthly': 30,
    'Quarterly': 90,
    'Every 3 Months': 90,
    'Annually': 365,
}

df['frequency_of_purchases'] = df['frequency_of_purchases'].map(freq_days)
print('Unmapped values:', df['frequency_of_purchases'].isnull().sum())
df['frequency_of_purchases'].value_counts().sort_index()

## 6. Create `age_group`
Four groups with roughly equal numbers of customers (quartiles of age).

In [ ]:
labels = ['Young Adult', 'Adult', 'Middle Aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)
df['age_group'] = df['age_group'].astype(str)

df.groupby('age_group')['age'].agg(['min', 'max', 'count'])

## 7. Final check

In [ ]:
print(df.shape)
print('Missing values:', df.isnull().sum().sum())
df.dtypes

In [ ]:
df.head()

### Optional: compare with the cleaned Excel file
Skip this cell if you do not have `Book1.xlsx` next to the notebook.

In [ ]:
import os
if os.path.exists('Book1.xlsx'):
    expected = pd.read_excel('Book1.xlsx')
    print('Same columns:', list(df.columns) == list(expected.columns))
    print('Same values :', df.equals(expected))
else:
    print('Book1.xlsx not found, skipping')

## 8. Save a cleaned copy

In [ ]:
df.to_csv('customer_shopping_cleaned.csv', index=False)

## 9. Load into SQL Server
Install once if needed: `pip install sqlalchemy pyodbc`

Change `SERVER` and `DATABASE` to your own. This uses Windows login (no password). Check your driver name with `pyodbc.drivers()`.

In [ ]:
from sqlalchemy import create_engine
import urllib

SERVER   = 'localhost'            # e.g. 'localhost' or 'DESKTOP-XXXX\\SQLEXPRESS'
DATABASE = 'YourDatabaseName'
DRIVER   = 'ODBC Driver 17 for SQL Server'

params = urllib.parse.quote_plus(
    f'DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;'
)
engine = create_engine(f'mssql+pyodbc:///?odbc_connect={params}')

df.to_sql('customer_shopping', engine, schema='dbo', if_exists='replace', index=False, chunksize=1000)
print('Loaded', len(df), 'rows into dbo.customer_shopping')